In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.Draw import MolsToGridImage

import sys
sys.path.append('../')
import FragShapley

/home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
plt.style.use('style.mplstyle')
sns.set_context('talk', font_scale=1.0)
fig_folder = os.path.abspath("figures/")

In [3]:
models = ['RF', 'GCN']#, 'MPNN']

In [4]:
# only select a single dataset
dataset_of_interest = 'SARS'
# consider dummy atoms
remove_dummies = False

files = [f'../5_potency/{model.lower()}_regression_potency/df_explanation.pkl' for model in models]
df_pot = pd.concat((pd.read_pickle(mut_file) for mut_file in files))

df_pot = df_pot.loc[df_pot.dataset == dataset_of_interest]

In [5]:
df_pot['fragments'] = df_pot.smiles.apply(lambda x: FragShapley.utils.get_BRICS_fragments_as_SMILES(x, remove_dummies=remove_dummies))

In [6]:
# get the values of the Explainer as a list (currently only available as dict)
df_pot['fragExplainer_shapley_values'] = df_pot.fragExplainer_result.apply(lambda x: list(x.values()))
# create a dataframe of all fragments (contains duplicates) and the corresponding Shapley Value
df_fragments = df_pot[['fragments', 'model', 'fragExplainer_shapley_values']].explode(['fragments', 'fragExplainer_shapley_values'], ignore_index=True)

In [7]:
df_fragments = df_fragments.groupby(['fragments', 'model']).agg([len, 'mean', 'std', list]) # will throw warning veacuse of error when calculating std for a single measurement
df_fragments = df_fragments.reset_index() # reset so that 'fragments' is a column again and no longer the index
df_fragments.columns = [col[0] if col[1]=='' else col[1] for col in df_fragments.columns.values] # rename, choose 'fragments' for first column, for the other simply len, mean, std
df_fragments

,fragments,model,len,mean,std,list
0,*C(*)=O,GCN,9,0.135497,0.114709,"[0.08212503592173263, -0.002354571554396087, 0..."
1,*C(*)=O,RF,9,0.012202,0.018669,"[0.003847759589947512, 0.00472315355725646, 0...."
2,*C(=O)C=C,GCN,3,-0.065035,0.059204,"[-0.13338632924216123, -0.029761427924746557, ..."
3,*C(=O)C=C,RF,3,-0.041844,0.080593,"[-0.08793296775793634, -0.08881435482804291, 0..."
4,*C(=O)[C@@H](*)C(C)(C)C,GCN,3,0.390044,0.000000,"[0.39004416919889906, 0.39004416919889906, 0.3..."
...,...,...,...,...,...,...
417,*n1cccn1,RF,1,-0.174884,NaN,[-0.17488395421476532]
418,*n1ccnn1,GCN,1,-0.058789,NaN,[-0.05878885587056479]
419,*n1ccnn1,RF,1,-0.209461,NaN,[-0.20946081613756662]
420,*n1ncnn1,GCN,1,0.085617,NaN,[0.08561677932739258]


In [8]:
from rdkit.Chem import Draw

do = Draw.MolDrawOptions()
do.legendFontSize = 32

min_n = 20
min_n = 5
mols_per_row = 5

top_k = 2 * mols_per_row

all_smiles = []
all_avg_contributions = []

for model in models:
    df_tmp = df_fragments.loc[df_fragments.model == model]
    df_tmp = df_tmp.sort_values(by='mean', ascending=False)
    df_tmp = df_tmp.loc[df_tmp.len >= min_n]
    smiles = df_tmp.iloc[:top_k].fragments.to_list()
    avg_contribution = df_tmp.iloc[:top_k]['mean'].to_list()
    all_smiles += smiles
    all_avg_contributions += avg_contribution
    print(f'{model}:')
    print('\t', smiles)
    print('\t', avg_contribution)

mols = [MolFromSmiles(sm) for sm in all_smiles]
svg = MolsToGridImage(mols=mols,
                      molsPerRow=mols_per_row,
                      legends=[f'{c:.2f}' for c in all_avg_contributions],
                      subImgSize=(200, 200),
                      drawOptions=do,
                      useSVG=True)
with open(os.path.join(fig_folder, "x_SARS_fragments.svg"), "w") as f:
    f.write(svg.data)

RF:
	 ['*N1C[C@@]2(C(=O)N(*)C[C@@H]2C)c2cc(Cl)ccc2C1=O', '*[C@H]1CN(*)C(=O)[C@@]12CN(*)C(=O)c1ccc(Cl)cc12', '*N1C[C@@]2(C(=O)N(*)C[C@@H]2C)c2cc(F)ccc2C1=O', '*N1C[C@]2(CCN(*)C2=O)c2cc(Cl)ccc2C1=O', '*c1cccc(Cl)c1', '*C1CCN(*)CC1', '*CC*', '*C(F)(F)F', '*C(C)C', '*C*']
	 [1.959893714497196, 1.6651924797663504, 1.311142399524152, 0.7810854111263372, 0.41599590209906023, 0.25471861141173685, 0.1562998283221398, 0.15447319532730236, 0.10518658614418151, 0.06104957847146329]
GCN:
	 ['*N1C[C@]2(CCN(*)C2=O)c2cc(Cl)ccc2C1=O', '*[C@H]1CN(*)C(=O)[C@@]12CN(*)C(=O)c1ccc(Cl)cc12', '*N1C[C@@]2(C(=O)N(*)C[C@@H]2C)c2cc(Cl)ccc2C1=O', '*C(F)(F)F', '*c1cccc(Cl)c1', '*N1C[C@@]2(C(=O)N(*)C[C@@H]2C)c2cc(F)ccc2C1=O', '*C(C)C', '*[C@H]1CCCN(*)C1', '*N*', '*O*']
	 [2.296543034106966, 2.2809092903743573, 2.2714369124379648, 1.5835233153940056, 1.3986212925596548, 0.7511168218794323, 0.5425672782791985, 0.5282235478361448, 0.4688387691974638, 0.45596450628742335]


In [19]:
# only correctly predicted molecules
df_plot = df_pot.copy()
df_plot['delta_pred'] = abs(df_plot.y_true - df_plot.y_pred)
df_plot = df_plot.loc[df_plot.delta_pred <= 0.5]
#df_plot = df_pot.query('abs(y_true - y_pred) <= 0.5')

# get number of fragments, more interesting to plot compounds with more than two fragments
df_plot['n_fragments'] = df_plot.fragments.apply(len)
df_plot = df_plot.loc[df_plot.n_fragments > 2]

In [20]:
model = 'RF'
#row_indices = {'RF': [3, 7, 9, 13, 20, 44]} # hand picked compounds
row_indices = {'RF': [0, 1, 2, 3, 4, 5]}

df_tmp = df_plot.loc[df_plot.model == model]
for idx in row_indices[model]:
    row = df_tmp.iloc[idx]
    contributions = FragShapley.visualization.get_atom_contribution_from_result_dict(row.smiles,
                                                                                     results_dict=row.fragExplainer_result,
                                                                                     frag_to_atom_ids=row.frag_to_atom_ids)
    out = FragShapley.visualization.visualize_contributions(smiles=row.smiles,
                                                            contributions=contributions,
                                                            scale=0.3,
                                                            color_pos=(0.5, 0.5, 1),
                                                            color_neg=(1, 0.5, 0.5),
                                                            legend=f'pred. pIC50: {row.y_pred:.2f}     exp. pIC50: {row.y_true:.2f}')
    with open(os.path.join(fig_folder, f"molecules_potency/x_potency_ex_mol_{model}_idx_{idx}.svg"), "w") as f:
        f.write(out.data)

In [21]:
df_tmp

,model,dataset,smiles,y_true,y_pred,fragExplainer_result,fragExplainer_expected_value,shap_result,shap_expected_value,atom_id_to_bits,frag_to_atom_ids,fragments,fragExplainer_shapley_values,delta_pred,n_fragments
6,RF,SARS,C[C@H]1CN(c2cncc3ccccc23)C(=O)[C@@]12CN(CC(=O)...,6.91,6.816401,"{0: 1.6284728287788615, 1: -0.3906339930555558...",5.678516,"[0.0, -0.0004357803914533065, -0.0010084829200...",5.252504,"{20: [41, 622, 807], 19: [80, 796, 1312], 23: ...","{0: [0, 1, 2, 3, 14, 15, 16, 17, 18, 30, 31, 3...",[*N1C[C@@]2(C(=O)N(*)C[C@@H]2C)c2cc(Cl)ccc2C1=...,"[1.6284728287788615, -0.3906339930555558, -0.0...",0.093599,6
8,RF,SARS,CN[C@H](C)CN1C[C@H](C(=O)Nc2cncc3ccccc23)c2cc(...,6.88,6.454544,"{0: 0.12393699718568349, 1: 0.0622605078763844...",5.678516,"[0.0, 0.011829927952430808, -0.001525442711072...",5.252504,"{2: [1, 1838, 2018], 12: [63, 1535, 1873], 4: ...","{0: [0, 1], 1: [2, 3, 4], 2: [5, 6, 7, 8, 9, 2...","[*NC, *C[C@H](*)C, *C(=O)[C@H]1CN(*)C(=O)c2ccc...","[0.12393699718568349, 0.062260507876384485, 0....",0.425456,5
12,RF,SARS,C[C@H]1CN(c2cncc3ccccc23)C(=O)[C@@]12CN(CC(=O)...,6.89,6.627512,"{0: 1.2214417557870376, 1: -0.4263396835317453...",5.678516,"[0.0, 1.1520138298237725e-05, -0.0010586695216...",5.252504,"{20: [41, 622, 807], 19: [80, 796, 1312], 23: ...","{0: [0, 1, 2, 3, 14, 15, 16, 17, 18, 28, 29, 3...",[*N1C[C@@]2(C(=O)N(*)C[C@@H]2C)c2cc(F)ccc2C1=O...,"[1.2214417557870376, -0.4263396835317453, 0.01...",0.262488,6
15,RF,SARS,CNC(=O)CN1C[C@@]2(C(=O)N(c3cncc4ccccc34)C[C@@H...,6.27,6.698047,"{0: 0.02654711127645566, 1: 0.0485345126713562...",5.678516,"[0.0, -0.00029041432538406297, -0.001901430514...",5.252504,"{2: [41, 807, 1805], 4: [80, 796, 1312], 23: [...","{0: [0, 1], 1: [2, 3, 4], 2: [5, 6, 7, 8, 9, 1...","[*NC, *CC(*)=O, *[C@H]1CN(*)C(=O)[C@@]12CN(*)C...","[0.02654711127645566, 0.048534512671356285, 1....",0.428047,6
22,RF,SARS,C[C@H]1CN(c2cncc3ccccc23)C(=O)[C@@]12CN(CCN1CC...,7.60,7.559732,"{0: 2.179315053210691, 1: -0.48894885912698394...",5.678516,"[0.0, -0.00037198972686155687, -0.001129105502...",5.252504,"{19: [80, 1145, 1548], 20: [80, 1145, 1441], 6...","{0: [0, 1, 2, 3, 14, 15, 16, 17, 18, 27, 28, 2...",[*N1C[C@@]2(C(=O)N(*)C[C@@H]2C)c2cc(Cl)ccc2C1=...,"[2.179315053210691, -0.48894885912698394, 0.18...",0.040268,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254,RF,SARS,O=C(Cc1cncc2ccccc12)N1CCC[C@H]2CCCC[C@H]21,5.15,5.152158,"{0: -0.02842442956349315, 1: -0.88165188988095...",5.678516,"[0.0, 0.0002479781780357371, -0.00261046934236...",5.252504,"{2: [80, 325, 817], 1: [119, 807, 1009], 5: [1...","{0: [0, 1, 2], 1: [3, 4, 5, 6, 7, 8, 9, 10, 11...","[*CC(*)=O, *c1cncc2ccccc12, *N1CCC[C@H]2CCCC[C...","[-0.02842442956349315, -0.8816518898809549, 0....",0.002158,3
255,RF,SARS,O=C(Cc1cncc2ccccc12)N1CCC[C@H]2CCCC[C@H]21,4.67,5.152158,"{0: -0.02842442956349315, 1: -0.88165188988095...",5.678516,"[0.0, 0.0002479781780357371, -0.00261046934236...",5.252504,"{2: [80, 325, 817], 1: [119, 807, 1009], 5: [1...","{0: [0, 1, 2], 1: [3, 4, 5, 6, 7, 8, 9, 10, 11...","[*CC(*)=O, *c1cncc2ccccc12, *N1CCC[C@H]2CCCC[C...","[-0.02842442956349315, -0.8816518898809549, 0....",0.482158,3
260,RF,SARS,O=C(Cc1cncc2ccccc12)N1CCC[C@H]2CCCC[C@@H]21,5.18,5.152158,"{0: -0.02842442956349315, 1: -0.88165188988095...",5.678516,"[0.0, 0.0002479781780357371, -0.00261046934236...",5.252504,"{2: [80, 325, 817], 1: [119, 807, 1009], 5: [1...","{0: [0, 1, 2], 1: [3, 4, 5, 6, 7, 8, 9, 10, 11...","[*CC(*)=O, *c1cncc2ccccc12, *N1CCC[C@H]2CCCC[C...","[-0.02842442956349315, -0.8816518898809549, 0....",0.027842,3
261,RF,SARS,COc1ccccc1[C@H]1C[C@H](C)CCN1C(=O)Cc1cncc2ccccc12,5.59,5.941248,"{0: 0.03945076471560978, 1: -0.061633826058201...",5.678516,"[0.0, 0.00035389549893693583, -0.0013939994042...",5.252504,"{13: [38, 926, 1480], 17: [80, 325, 817], 15: ...","{0: [0, 1], 1: [2, 3, 4, 5, 6, 7], 2: [8, 9, 1...","[*OC, *c1ccccc1*, *[C@H]1C[C@H](C)CCN1*, *CC(*...","[0.03945076471560978, -0.061633826058201696, 0...",0.351248,5
